# Push GliZNet to Hugging Face Hub

Loads a local checkpoint and pushes model + tokenizer to `alexneakameni/gliznet-deberta-v3-base`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
from huggingface_hub import HfApi

from gliznet.model import GliZNetForSequenceClassification
from gliznet.tokenizer import GliZNETTokenizer

# ── Configuration ─────────────────────────────────────────────────────────────
CHECKPOINT   = "../results/deberta-v3-base_20260429_100035/checkpoint-7056"
REPO_ID      = "alexneakameni/gliznet-deberta-v3-base"
PRIVATE      = False   # set True to keep the repo private until ready
# ──────────────────────────────────────────────────────────────────────────────

print(f"Checkpoint : {CHECKPOINT}")
print(f"Target repo: {REPO_ID}")

Checkpoint : ../results/deberta-v3-base_20260429_100035/checkpoint-7056
Target repo: alexneakameni/gliznet-deberta-v3-base


In [2]:
from huggingface_hub import login, whoami

# Login via token stored in ~/.huggingface/token (set once with `huggingface-cli login`)
# or pass token= explicitly: login(token="hf_...")
login()
user = whoami()
print(f"✓ Logged in as: {user['name']}")

✓ Logged in as: alexneakameni


In [3]:
model = GliZNetForSequenceClassification.from_pretrained(CHECKPOINT)
tokenizer = GliZNETTokenizer.from_pretrained(CHECKPOINT)

print(f"✓ Model loaded  — {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"✓ Tokenizer loaded — vocab size: {len(tokenizer)}")
print(f"  dtype: {next(model.parameters()).dtype}")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✓ Model loaded  — 184,421,377 parameters
✓ Tokenizer loaded — vocab size: 128002
  dtype: torch.float32


In [4]:
# Create the repo if it doesn't exist yet
api = HfApi()
repo_url = api.create_repo(
    repo_id=REPO_ID,
    private=PRIVATE,
    exist_ok=True,   # no-op if already exists
)
print(f"✓ Repo ready: {repo_url}")

✓ Repo ready: https://huggingface.co/alexneakameni/gliznet-deberta-v3-base


In [5]:
print("Pushing model weights…")
model.push_to_hub(
    REPO_ID,
    commit_message="Upload GliZNet DeBERTa-v3-base checkpoint",
    private=PRIVATE,
)

print("Pushing tokenizer…")
tokenizer.push_to_hub(
    REPO_ID,
    commit_message="Upload GliZNET tokenizer",
    private=PRIVATE,
)

print(f"\n✓ Done! Model available at: https://huggingface.co/{REPO_ID}")

Pushing model weights…


README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushing tokenizer…


No files have been modified since last commit. Skipping to prevent empty commit.



✓ Done! Model available at: https://huggingface.co/alexneakameni/gliznet-deberta-v3-base


## Verify — reload from Hub

Load the model back from the Hub and run a quick sanity prediction.

In [6]:
from gliznet.predictor import ZeroShotClassificationPipeline

pipeline = ZeroShotClassificationPipeline.from_pretrained(
    REPO_ID,
    device="cuda" if torch.cuda.is_available() else "cpu",
)
text   = "Scientists discover a new exoplanet orbiting a distant star."
labels = ["astronomy", "politics", "cooking", "space exploration", "finance"]

result = pipeline(text, labels)
print(f"Text: {text}\n")
for ls in sorted(result.labels, key=lambda x: -x.score):
    bar = "█" * int(ls.score * 20)
    print(f"  {ls.label:<25} {ls.score:.3f}  {bar}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/738M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Text: Scientists discover a new exoplanet orbiting a distant star.

  astronomy                 1.000  ███████████████████
  space exploration         0.999  ███████████████████
  finance                   0.000  
  politics                  0.000  
  cooking                   0.000  
